# Multivariate Analysis

**Topic:** Exploratory Data Analysis

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns

import plotly.graph_objects as go
import plotly.express as px

import ipywidgets as widgets
from ipywidgets import Dropdown, SelectMultiple, Output, HBox, VBox, HTML

from IPython.display import display, clear_output
from scipy import stats

titanic = sns.load_dataset("titanic")
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this session you will be able to:

- **Interpret** a correlation heatmap to identify pairs of variables with strong positive or negative relationships
- **Explain** what a correlation coefficient near 0, near 1, and near -1 each means for two variables
- **Use** grouped visualizations to see how a third variable changes a bivariate relationship

> **Tip:** In the heatmap widget, deselect `pclass` and see how the remaining correlations change. In the grouped analysis widget, try `pclass` on the x-axis with `sex` as the color grouping and a bar chart. That combination reveals the story inside the story.

---
## How we got here

In `09_bivariate_analysis.ipynb` we compared pairs of variables one at a time. Now we look at the full picture: all numeric variables at once using a correlation matrix, and grouped analyses that introduce a third variable to reveal patterns hidden inside bivariate comparisons.

---
## Why this matters for data science

Multivariate analysis catches two things bivariate analysis misses. First, it reveals multicollinearity: when two features are highly correlated with each other, adding both to a linear model causes instability. Second, it reveals interaction effects: a relationship that looks weak overall might be strong within a specific subgroup. Both findings directly shape which features you include in a model.

---
## Try it yourself

In [2]:
out_heatmap = Output()
out_grouped = Output()

numeric_cols = titanic.select_dtypes(include="number").columns.tolist()
usable_cols = [c for c in titanic.columns if c not in ["name", "ticket", "cabin", "boat", "body", "home.dest"]]

col_multiselect = SelectMultiple(
    options=numeric_cols,
    value=numeric_cols[:5],
    description="Columns:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="280px", height="160px"),
)

x_dropdown = Dropdown(
    options=usable_cols,
    value="pclass",
    description="X axis:",
    style={"description_width": "70px"},
    layout=widgets.Layout(width="260px"),
)

color_dropdown = Dropdown(
    options=[c for c in usable_cols if titanic[c].nunique() <= 8],
    value="sex",
    description="Color by:",
    style={"description_width": "70px"},
    layout=widgets.Layout(width="260px"),
)

chart_dropdown = Dropdown(
    options=["Bar chart", "Box plot", "Scatter"],
    value="Bar chart",
    description="Chart type:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="260px"),
)

def render_heatmap(selected_cols):
    if len(selected_cols) < 2:
        with out_heatmap:
            clear_output(wait=True)
            display(HTML('<em>Select at least 2 columns to show a correlation heatmap.</em>'))
        return
    corr = titanic[list(selected_cols)].corr()
    n = len(selected_cols)
    fig = go.Figure(
        data=go.Heatmap(
            z=corr.values,
            x=list(selected_cols),
            y=list(selected_cols),
            colorscale=[
                [0.0, "#F76707"], [0.5, "#FFFFFF"], [1.0, "#4C6EF5"]
            ],
            zmid=0,
            text=[[f"{v:.2f}" for v in row] for row in corr.values],
            texttemplate="%{text}",
            textfont={"size": 12},
            hoverongaps=False,
        ),
    )
    layout = base_layout(title="Correlation Heatmap (Pearson r)")
    layout.update(height=max(320, n * 60 + 120))
    fig.update_layout(layout)
    fig.update_layout(yaxis_autorange="reversed")
    with out_heatmap:
        clear_output(wait=True)
        display(go.FigureWidget(fig))

def render_grouped(x_col, color_col, chart_type):
    df = titanic[[x_col, color_col]].dropna()
    groups = df[color_col].unique()
    colors_list = [PALETTE["primary"], PALETTE["secondary"], PALETTE["accent"], PALETTE["muted"]]

    if chart_type == "Bar chart":
        pivot = df.groupby([x_col, color_col]).size().unstack(fill_value=0)
        traces = [
            go.Bar(
                name=str(g),
                x=[str(v) for v in pivot.index],
                y=pivot[g].values if g in pivot.columns else [],
                marker_color=colors_list[i % len(colors_list)],
            )
            for i, g in enumerate(groups)
        ]
        layout = base_layout(title=f"{x_col} by {color_col} — Grouped Bar", xaxis_title=x_col, yaxis_title="Count")
        layout.update(barmode="group")
    elif chart_type == "Box plot":
        numeric_candidates = titanic.select_dtypes(include="number").columns.tolist()
        y_col = numeric_candidates[0] if x_col not in numeric_candidates else x_col
        df2 = titanic[[y_col, color_col]].dropna()
        traces = [
            go.Box(
                x=df2.loc[df2[color_col] == g, y_col],
                name=str(g),
                marker_color=colors_list[i % len(colors_list)],
            )
            for i, g in enumerate(df2[color_col].unique())
        ]
        layout = base_layout(title=f"{y_col} by {color_col} — Box Plot", xaxis_title=y_col, yaxis_title="")
    else:
        numeric_candidates = titanic.select_dtypes(include="number").columns.tolist()
        if x_col in numeric_candidates and len(numeric_candidates) >= 2:
            y_col = [c for c in numeric_candidates if c != x_col][0]
        else:
            y_col = numeric_candidates[0]
        df2 = titanic[[x_col, y_col, color_col]].dropna()
        traces = [
            go.Scatter(
                x=df2.loc[df2[color_col] == g, x_col],
                y=df2.loc[df2[color_col] == g, y_col],
                mode="markers",
                name=str(g),
                marker=dict(color=colors_list[i % len(colors_list)], opacity=0.5, size=6),
            )
            for i, g in enumerate(df2[color_col].unique())
        ]
        layout = base_layout(title=f"{x_col} vs {y_col} by {color_col}", xaxis_title=x_col, yaxis_title=y_col)

    layout.update(showlegend=True, height=380)
    fig = go.Figure(data=traces, layout=layout)
    with out_grouped:
        clear_output(wait=True)
        display(go.FigureWidget(fig))

def on_heatmap_change(change):
    render_heatmap(col_multiselect.value)

def on_grouped_change(change):
    render_grouped(x_dropdown.value, color_dropdown.value, chart_dropdown.value)

col_multiselect.observe(on_heatmap_change, names="value")
x_dropdown.observe(on_grouped_change, names="value")
color_dropdown.observe(on_grouped_change, names="value")
chart_dropdown.observe(on_grouped_change, names="value")

display(HTML('<strong style="font-family:Inter,Arial,sans-serif;">Part 1: Correlation Heatmap</strong>'))
display(VBox([
    HTML('<em style="font-family:Inter,Arial,sans-serif;font-size:13px;">Hold Ctrl/Cmd to select multiple columns.</em>'),
    col_multiselect,
    out_heatmap,
]))
display(HTML('<br><strong style="font-family:Inter,Arial,sans-serif;">Part 2: Grouped Analysis</strong>'))
display(VBox([HBox([x_dropdown, color_dropdown, chart_dropdown]), out_grouped]))

render_heatmap(col_multiselect.value)
render_grouped(x_dropdown.value, color_dropdown.value, chart_dropdown.value)

HTML(value='<strong style="font-family:Inter,Arial,sans-serif;">Part 1: Correlation Heatmap</strong>')

HTML(value='<br><strong style="font-family:Inter,Arial,sans-serif;">Part 2: Grouped Analysis</strong>')

---
## What's happening?

A correlation matrix shows the Pearson r coefficient for every pair of numeric variables in the dataset. The values range from -1 to +1:

| Value | Meaning |
|-------|---------|
| r = 1.0 | Perfect positive linear relationship |
| r = 0.7 | Strong positive linear relationship |
| r = 0.3 | Weak positive linear relationship |
| r ≈ 0.0 | No linear relationship |
| r = -0.3 | Weak negative linear relationship |
| r = -0.7 | Strong negative linear relationship |

The diagonal is always 1.0 (a variable is perfectly correlated with itself). The matrix is symmetric: the value at row A, column B equals the value at row B, column A.

Correlation only measures linear relationships. Two variables can have a strong non-linear relationship and still show r ≈ 0.

---
## Real-world example: Titanic Dataset

The full correlation matrix below shows all pairwise relationships among the numeric Titanic columns.

Notice:
- **`pclass` and `survived`** have a moderate negative correlation (r ≈ -0.34). Higher class number (3rd) is associated with lower survival
- **`pclass` and `fare`** have a strong negative correlation (r ≈ -0.55). Lower class number (1st) paid more
- **`age` and `fare`** show a slight positive correlation (r ≈ 0.10), which is weak but consistent

> **Discussion question:** Age and fare are slightly positively correlated. What real-world mechanism might explain that? Think about who could afford first-class travel in 1912.

In [3]:
numeric_cols_hm = ["survived", "pclass", "age", "sibsp", "parch", "fare"]
corr_matrix = titanic[numeric_cols_hm].corr()

fig = go.Figure(
    data=go.Heatmap(
        z=corr_matrix.values,
        x=numeric_cols_hm,
        y=numeric_cols_hm,
        colorscale=[[0.0, "#F76707"], [0.5, "#FFFFFF"], [1.0, "#4C6EF5"]],
        zmid=0,
        text=[[f"{v:.2f}" for v in row] for row in corr_matrix.values],
        texttemplate="%{text}",
        textfont={"size": 12},
    ),
)

layout = base_layout(
    title="Titanic Numeric Variable Correlation Matrix (Pearson r)",
)
layout.update(height=420)
fig.update_layout(layout)
fig.update_layout(yaxis_autorange="reversed")
fig.show()

### Correlation patterns and what they mean for modeling

| Finding | Implication |
|---------|-------------|
| Two features with r > 0.9 | Drop one — they carry the same information |
| Feature with |r| > 0.3 to target | Likely useful predictor — explore further |
| Feature with r ≈ 0.0 to all others | May still be useful non-linearly — test in model |
| All features weakly correlated to target | Problem may be non-linear — try tree models |
| Unexpected strong correlation | Check for data leakage — target info in the feature? |

---
## Key takeaway

> **A correlation matrix does not tell you which variables to keep. It tells you which variables share information, and sharing too much information between features is just as problematic as having too little.**

---
*Next up: 11_feature_distributions.ipynb — examining whether each feature follows a normal distribution and preparing features for modeling*